# FreeFine Final Geometry — Per-image SUBC / BGC / WRAP_E on the full 5,677 (T4×2)

**Purpose.** The final metric table reports group means only. This notebook re-scores the
*same* final images with the *official* FreeFine metric code, patched only to also write the
value of every individual image. That enables paired statistics between SGR-EPSREC and the
baseline: bootstrap confidence intervals, Wilcoxon signed-rank tests and win/tie/loss counts.

Nothing is regenerated. MD and the FID family are **not** run (MD is the 6-hour metric and is
not a paired statistic; the FID family has no per-image value).

**Built-in checks — the notebook stops if any fails:**
1. every shard folder has the expected number of PNGs and the benchmark fingerprint matches;
2. rotation images are byte-identical between the baseline and SGR-EPSREC;
3. the per-image values average *exactly* to the value the official code prints in the same run;
4. the group means reproduce the final metric table of the thesis to 4 decimal places.

### Inputs to attach (7)
1. `01-full5677-generation-shard0of4-t4x2-v1` (account georgiostzamouranis)
2. `01-full5677-generation-shard1of4-t4x2-v1` (account giorgostzam)
3. `01-full5677-generation-shard2of4-t4x2-v1` (account papadonikolas)
4. `01-full5677-generation-shard3of4-t4x2-v1` (account gtz19800)
5. dataset `freefine-sample-metadata`
6. dataset `geobench2d-coarse-img`
7. dataset `geobench2d-metrics-subset`

Shortcut: forking `fork-of-09-combined-metric-resume-accounta-t4x2-v2` inherits all seven.

### Settings
Accelerator **GPU T4 ×2**, Internet **ON**. Then *Run All*.
Expected time: ~10 min environment + ~30–45 min metrics.

### Outputs (download these three)
- `percase_metrics_full5677.csv` — one row per edit, with metadata and every per-image value
- `paired_stats.csv` — paired statistics per group and metric
- `percase_validation_report.json` — the numbers behind checks 1–4


In [ ]:
# ===== 1. Configuration, clone and pin the FreeFine source =====
import os, time, subprocess
NB_START = time.time()
COMMIT = "4c9fdb971572b32edbeac13464659274c28decbb"   # same pin as the final generation and metric runs
WORK = "/kaggle/working"
INCLUDE_MIDHF_SEVERE = True   # also score SGR-MIDHF on the 759 severe resizes, the only cases where it differs

subprocess.run(
    f"mkdir -p /kaggle/temp && cd /kaggle/temp && rm -rf FreeFine && "
    f"git clone -q https://github.com/CIawevy/FreeFine.git && "
    f"cd FreeFine && git checkout -q {COMMIT}",
    shell=True, check=True)
print("✓ cloned + pinned FreeFine", COMMIT[:14])


In [ ]:
%%bash
# ===== 2. Metric environment (identical to the final metric runs) =====
set -e
pip install -q --root-user-action=ignore uv
uv python install 3.10.13
V=/kaggle/temp/metric_env
PY=$V/bin/python
REPO=/kaggle/temp/FreeFine
rm -rf "$V"
uv venv --python 3.10.13 "$V"
uv pip install --python "$PY" torch==2.6.0 torchvision==0.21.0 torchaudio==2.6.0 --index-url https://download.pytorch.org/whl/cu124
uv pip install --python "$PY" "setuptools<70" wheel pip
grep -vi '^clip' $REPO/evaluation/metrics/requirements.txt > /tmp/m.txt
uv pip install --python "$PY" -r /tmp/m.txt
uv pip install --python "$PY" "setuptools<70"
uv pip install --python "$PY" --no-build-isolation "clip @ git+https://github.com/openai/CLIP.git@dcba3cb2e2827b402d2701e7e1c7d9fed8a20ef1"
uv pip install --python "$PY" "pyarrow<16" "datasets<3"
wget -q https://dl.fbaipublicfiles.com/mmf/clip/bpe_simple_vocab_16e6.txt.gz -P /tmp
for d in $(find $V -path '*/site-packages/clip' -o -path '*open_clip' -type d); do
  cp /tmp/bpe_simple_vocab_16e6.txt.gz "$d/" 2>/dev/null || true
done
echo "✓ metric_env OK"


In [ ]:
# ===== 3. Official metric patches + per-image dump =====
# The per-image patches only record the value each official loop already computes; the
# accumulation of the group mean is untouched, and check 3 below verifies that.
import os, pathlib, py_compile
MR = pathlib.Path(os.environ.get("FF_METRICS_DIR", "/kaggle/temp/FreeFine/evaluation/metrics"))

def patch(path, old, new, marker):
    s = path.read_text(encoding="utf-8")
    if marker in s:                      # already patched (cell re-run)
        return
    assert s.count(old) == 1, f"anchor not found exactly once in {path.name}: {old!r}"
    path.write_text(s.replace(old, new, 1), encoding="utf-8")

# (a) the fixes used by every official metric run in this project
patch(MR / "main.py", "args.3d", "getattr(args,'3d')", "getattr(args,'3d')")
patch(MR / "main.py",
      "    args = parser.parse_args()\n",
      "    args = parser.parse_args()\n"
      "    import random as _random, numpy as _np\n"
      "    try:\n"
      "        import torch as _torch\n"
      "    except Exception:\n"
      "        _torch=None\n"
      "    _metric_seed=int(os.environ.get('FF_METRIC_SEED','42'))\n"
      "    _random.seed(_metric_seed); _np.random.seed(_metric_seed)\n"
      "    if _torch is not None:\n"
      "        _torch.manual_seed(_metric_seed)\n"
      "        if _torch.cuda.is_available(): _torch.cuda.manual_seed_all(_metric_seed)\n",
      "FF_METRIC_SEED")
for f in [MR / "MD" / "mean_distance.py", MR / "MD" / "dift_sd.py"]:   # MD is not run; kept for parity
    f.write_text(f.read_text(encoding="utf-8").replace("stabilityai/stable-diffusion-2-1", "sd2-community/stable-diffusion-2-1"), encoding="utf-8")

# (b) the per-image writer; a no-op unless FF_PERCASE_PREFIX is set
(MR / "ff_percase.py").write_text(
    "import os, csv\n"
    "\n"
    "def dump(metric, rows):\n"
    "    prefix = os.environ.get('FF_PERCASE_PREFIX')\n"
    "    if not prefix:\n"
    "        return\n"
    "    path = prefix + '_' + metric + '.csv'\n"
    "    os.makedirs(os.path.dirname(path), exist_ok=True)\n"
    "    with open(path, 'w', newline='') as f:\n"
    "        w = csv.writer(f)\n"
    "        w.writerow(['gen_path', metric])\n"
    "        for p, v in rows:\n"
    "            w.writerow([p, repr(float(v))])\n"
    "    print('per-image %s: %d rows -> %s' % (metric, len(rows), path), flush=True)\n")

# (c) record each per-image value inside the three official loops
patch(MR / "VBench" / "subject_consistency.py",
      "    for inputs in tqdm(data_pair):\n"
      "        subc += compute_subject_consistency(inputs, dino_model, device)\n"
      "    subc = subc/len(data_pair)\n",
      "    _per = []\n"
      "    for inputs in tqdm(data_pair):\n"
      "        _v = compute_subject_consistency(inputs, dino_model, device)\n"
      "        subc += _v\n"
      "        _per.append((inputs[1], _v))\n"
      "    subc = subc/len(data_pair)\n"
      "    import ff_percase; ff_percase.dump('SUBC', _per)\n",
      "ff_percase")
patch(MR / "VBench" / "background_consistency.py",
      "    for inputs in tqdm(data_pair):\n"
      "        bgc += compute_background_consistency(inputs, clip_model, preprocess, device)\n"
      "    bgc = bgc/len(data_pair)\n",
      "    _per = []\n"
      "    for inputs in tqdm(data_pair):\n"
      "        _v = compute_background_consistency(inputs, clip_model, preprocess, device)\n"
      "        bgc += _v\n"
      "        _per.append((inputs[1], _v))\n"
      "    bgc = bgc/len(data_pair)\n"
      "    import ff_percase; ff_percase.dump('BGC', _per)\n",
      "ff_percase")
we = MR / "wrap_error.py"
patch(we, "    num = 0\n", "    num = 0\n    _per = []\n", "_per = []")
patch(we,
      "                wrap_e += np.sum(np.abs(images[0]*mask - images[1]*mask)) / mask.sum()\n",
      "                _v = np.sum(np.abs(images[0]*mask - images[1]*mask)) / mask.sum()\n"
      "                wrap_e += _v\n"
      "                _per.append((sample[image_label], float(_v)))\n",
      "wrap_e += _v")
patch(we, "    wrap_e = wrap_e/num\n",
      "    wrap_e = wrap_e/num\n    import ff_percase; ff_percase.dump('WRAP_E', _per)\n",
      "ff_percase")

for f in [MR / "main.py", MR / "ff_percase.py", MR / "wrap_error.py",
          MR / "VBench" / "subject_consistency.py", MR / "VBench" / "background_consistency.py",
          MR / "MD" / "mean_distance.py", MR / "MD" / "dift_sd.py"]:
    py_compile.compile(str(f), doraise=True)
print("✓ official metric patches + per-image dump installed")


In [ ]:
# ===== 4. PyTorch 2.6 / OpenAI CLIP compatibility (identical to the final metric run) =====
import glob, pathlib
_clip_py = glob.glob("/kaggle/temp/metric_env/lib/python3.10/site-packages/clip/clip.py")
assert len(_clip_py) == 1, _clip_py
_cp = pathlib.Path(_clip_py[0]); _cs = _cp.read_text(encoding="utf-8")
_old = 'state_dict = torch.load(opened_file, map_location="cpu")'
_new = 'state_dict = torch.load(opened_file, map_location="cpu", weights_only=False)'
if _old in _cs:
    _cp.write_text(_cs.replace(_old, _new, 1), encoding="utf-8")
elif _new not in _cs:
    raise AssertionError("Unexpected OpenAI CLIP loader source; refusing broad torch.load patch")
print("✓ OpenAI CLIP PyTorch-2.6 compatibility patch installed")


In [ ]:
# ===== 5. Assemble the exact 5,677-image trees from the four generation shards =====
import os, glob, json, csv, shutil, math, hashlib, filecmp
from collections import defaultdict, Counter

EXPECTED_SHA = "17d971102daca232921d33de91adac58ef0ddbea9c9f814e219148486c9de802"
MODELS = {"FREEFINE_BASELINE": "baseline", "SGR_EPSREC": "SGR_EPSREC"}
if INCLUDE_MIDHF_SEVERE:
    MODELS["SGR_MIDHF_EPSREC"] = "SGR_MIDHF_EPSREC"

GEO = "/kaggle/temp/GeoBenchMeta"
os.makedirs(f"{GEO}/Geo-Bench-2D", exist_ok=True)

def one(pattern, desc):
    xs = glob.glob(pattern, recursive=True)
    if not xs:
        raise FileNotFoundError(f"Missing {desc}: {pattern}")
    return sorted(xs, key=lambda x: (len(x), x))[0]

CACHE = next(c for c in glob.glob("/kaggle/input/**/Geo-Bench-2D", recursive=True)
             if os.path.isdir(f"{c}/source_img"))
COARSE = one("/kaggle/input/**/coarse_img/*/*/*.png", "coarse_img").split("/coarse_img/")[0] + "/coarse_img"
ANNP = one("/kaggle/input/**/annotation_2d.json", "annotation_2d.json")
META = one("/kaggle/input/**/sample_metadata.csv", "sample_metadata.csv")

for nm in ["source_img", "source_mask", "target_mask", "source_img_full_v2"]:
    d = f"{GEO}/Geo-Bench-2D/{nm}"
    if os.path.lexists(d):
        os.remove(d) if os.path.islink(d) else shutil.rmtree(d)
    os.symlink(f"{CACHE}/{nm}", d)
d = f"{GEO}/Geo-Bench-2D/coarse_img"
if os.path.lexists(d):
    os.remove(d) if os.path.islink(d) else shutil.rmtree(d)
os.symlink(COARSE, d)
shutil.copy(ANNP, f"{GEO}/annotation_2d.json")
ann = json.load(open(f"{GEO}/annotation_2d.json"))

meta = []
for r in csv.DictReader(open(META)):
    if r["edit_type"] not in ("move", "rotate", "resize"):
        continue
    d, i, e = str(r["da_n"]), str(r["ins_id"]), str(r["case_id"])
    ep = ann[d]["instances"][i][e]["edit_param"]
    scale = math.sqrt(float(ep[6]) * float(ep[7]))
    rr = dict(r)
    rr.update({"da_n": d, "ins_id": i, "case_id": e, "scale_exact": scale,
               "difficulty": str(r.get("difficulty", "")).strip().lower(),
               "affine_severe": (r["edit_type"] == "resize" and (scale >= 1.5 or scale <= 0.6))})
    meta.append(rr)
assert len(meta) == 5677
assert Counter(r["edit_type"] for r in meta) == {"resize": 2635, "rotate": 1603, "move": 1439}
assert sum(r["affine_severe"] for r in meta) == 759

lines = []
for r in sorted(meta, key=lambda z: (z["da_n"], z["ins_id"], z["case_id"])):
    vals = [float(x) for x in ann[r["da_n"]]["instances"][r["ins_id"]][r["case_id"]]["edit_param"]]
    lines.append(f"{r['da_n']}|{r['ins_id']}|{r['case_id']}|{r['edit_type']}|"
                 + "|".join(f"{x:.12g}" for x in vals))
sha = hashlib.sha256("\n".join(lines).encode()).hexdigest()
assert sha == EXPECTED_SHA, (sha, EXPECTED_SHA)
print("✓ exact benchmark fingerprint", sha)

# Deterministic shard assignment used by the final generation run.
def category(r):
    if r["edit_type"] == "move":
        return "move"
    if r["edit_type"] == "rotate":
        return "rotate"
    return "resize_severe" if r["affine_severe"] else "resize_nonsevere"
offset = {"move": 0, "rotate": 1, "resize_nonsevere": 0, "resize_severe": 2}
bycat = defaultdict(list)
for r in meta:
    bycat[category(r)].append(r)
row_shard = {}
for cat, rows in bycat.items():
    for j, r in enumerate(sorted(rows, key=lambda z: (z["da_n"], z["ins_id"], z["case_id"]))):
        row_shard[(r["da_n"], r["ins_id"], r["case_id"])] = (j + offset[cat]) % 4

def pc(p):
    return len(glob.glob(p + "/**/*.png", recursive=True)) if os.path.isdir(p) else 0

EXPECTED_SHARD_PNGS = {0: 1419, 1: 1419, 2: 1420, 3: 1419}
ROOTS = {}
for slug, model_dir in MODELS.items():
    ROOTS[slug] = {}
    for s in range(4):
        cands = glob.glob(f"/kaggle/input/**/final_geometry_full/shard_{s}/{model_dir}", recursive=True)
        if not cands:
            raise FileNotFoundError(
                f"Missing generation shard {s} for {slug}: attach 01-full5677-generation-shard{s}of4-t4x2-v1")
        root = max(cands, key=pc)
        n = pc(root)
        print(f"{slug:18s} shard {s}: {n} pngs  {root}")
        assert n == EXPECTED_SHARD_PNGS[s], f"{slug} shard {s} has {n} pngs, expected {EXPECTED_SHARD_PNGS[s]}"
        ROOTS[slug][s] = root

EVALROOTS = {}
for slug in MODELS:
    ev = f"{GEO}/gen_eval/{slug}"
    if os.path.exists(ev):
        shutil.rmtree(ev)
    missing = []
    for r in meta:
        k = (r["da_n"], r["ins_id"], r["case_id"])
        src = f"{ROOTS[slug][row_shard[k]]}/{r['da_n']}/{r['ins_id']}/{r['case_id']}.png"
        dst = f"{ev}/{r['da_n']}/{r['ins_id']}/{r['case_id']}.png"
        if not os.path.exists(src):
            missing.append(src)
            continue
        os.makedirs(os.path.dirname(dst), exist_ok=True)
        os.symlink(os.path.realpath(src), dst)
    assert not missing, missing[:10]
    EVALROOTS[slug] = ev
    print("✓ assembled", slug, "5677/5677")

def keys(pred):
    return [(r["da_n"], r["ins_id"], r["case_id"]) for r in meta if pred(r)]

GROUPS = {
    "all_5677": keys(lambda r: True),
    "move_1439": keys(lambda r: r["edit_type"] == "move"),
    "rotate_1603": keys(lambda r: r["edit_type"] == "rotate"),
    "resize_2635": keys(lambda r: r["edit_type"] == "resize"),
    "resize_severe_759": keys(lambda r: r["edit_type"] == "resize" and r["affine_severe"]),
    "resize_nonsevere_1876": keys(lambda r: r["edit_type"] == "resize" and not r["affine_severe"]),
}
for lvl in ["easy", "medium", "hard"]:
    GROUPS[f"diff_{lvl}"] = keys(lambda r, l=lvl: r["difficulty"] == l)
    for t in ["move", "rotate", "resize"]:
        GROUPS[f"{t}_{lvl}"] = keys(lambda r, l=lvl, t=t: r["difficulty"] == l and r["edit_type"] == t)
print({g: len(v) for g, v in GROUPS.items()})

# Cross-check against the stratification table of the thesis (warning only; does not stop the run).
EXPECTED_CELLS = {"move_easy": 484, "move_medium": 542, "move_hard": 413,
                  "rotate_easy": 434, "rotate_medium": 489, "rotate_hard": 680,
                  "resize_easy": 974, "resize_medium": 861, "resize_hard": 800}
bad = {g: (len(GROUPS[g]), n) for g, n in EXPECTED_CELLS.items() if len(GROUPS[g]) != n}
print("✓ difficulty cells match the thesis crosstab" if not bad
      else f"⚠ difficulty cells differ from the thesis crosstab: {bad}")

# Rotation is a byte-identical copy of the baseline in SGR-EPSREC; verify before relying on it.
for d, i, e in GROUPS["rotate_1603"]:
    pa = f"{EVALROOTS['FREEFINE_BASELINE']}/{d}/{i}/{e}.png"
    pb = f"{EVALROOTS['SGR_EPSREC']}/{d}/{i}/{e}.png"
    assert filecmp.cmp(pa, pb, shallow=False), f"rotate case not identical: {d}/{i}/{e}"
print("✓ rotate_1603 byte-identical between FREEFINE_BASELINE and SGR_EPSREC")


In [ ]:
# ===== 6. Download CLIP and DINO once, serially, before the two GPU workers start =====
import subprocess, os
PY = "/kaggle/temp/metric_env/bin/python"
ENV_BASE = os.environ.copy()
ENV_BASE.update({"HF_HOME": "/kaggle/temp/hf", "TORCH_HOME": "/kaggle/temp/torch",
                 "TOKENIZERS_PARALLELISM": "false", "MPLBACKEND": "Agg"})
code_str = ("import clip, torch\n"
            "print('torch', torch.__version__)\n"
            "clip.load('ViT-B/32', device='cpu', jit=False)\n"
            "print('CLIP_PREWARM_OK')\n"
            "torch.hub.load('facebookresearch/dino:main', 'dino_vitb16')\n"
            "print('DINO_PREWARM_OK')\n")
r = subprocess.run([PY, "-c", code_str], env=ENV_BASE, capture_output=True, text=True)
print(r.stdout)
if r.returncode != 0:
    print(r.stderr[-5000:])
    raise RuntimeError("model prewarm failed; do not start the metric workers")
print("✓ CLIP and DINO cached; safe to start GPU workers")


In [ ]:
# ===== 7. Official BGC / SUBC / WRAP_E with the per-image dump, one job per GPU =====
import os, json, re, subprocess, threading, queue, time

MET = "/kaggle/temp/FreeFine/evaluation/metrics"
OUT = f"{WORK}/percase"
LOG = f"{WORK}/percase_logs"
os.makedirs(OUT, exist_ok=True)
os.makedirs(LOG, exist_ok=True)

# Order of the task string in main.py: FID, IRS, HPS, BGC, SUBC, WRAP_E, MD, FID_DINO, FID_KD
TASK = "000111000"
PER_METRICS = ["SUBC", "BGC", "WRAP_E"]

JOBS = [("FREEFINE_BASELINE", "all_5677"), ("SGR_EPSREC", "all_5677")]
if INCLUDE_MIDHF_SEVERE:
    JOBS.append(("SGR_MIDHF_EPSREC", "resize_severe_759"))

def manifest(slug, gname):
    o = {}
    for d, i, e in GROUPS[gname]:
        lf = dict(ann[d]["instances"][i][e])
        lf["gen_img_path"] = f"gen_eval/{slug}/{d}/{i}/{e}.png"
        o.setdefault(d, {"instances": {}})["instances"].setdefault(i, {})[e] = lf
    p = f"{GEO}/m_{slug}_{gname}.json"
    json.dump(o, open(p, "w"))
    return p

def done(slug, gname):
    n = len(GROUPS[gname])
    for m in PER_METRICS:
        p = f"{OUT}/{slug}__{gname}_{m}.csv"
        if not os.path.exists(p) or sum(1 for _ in open(p)) != n + 1:
            return False
    return os.path.exists(f"{OUT}/{slug}__{gname}_official.json")

def run(gpu, slug, gname):
    env = ENV_BASE.copy()
    env.update({"CUDA_VISIBLE_DEVICES": str(gpu), "FF_METRIC_SEED": "42", "FF_MD_SEED": "42",
                "PYTORCH_CUDA_ALLOC_CONF": "expandable_segments:True",
                "FF_PERCASE_PREFIX": f"{OUT}/{slug}__{gname}"})
    t0 = time.time()
    p = subprocess.run([PY, "main.py", "--path", manifest(slug, gname), "--use_relative_path",
                        "--base_dir", GEO, "--fid_path", f"{GEO}/Geo-Bench-2D/source_img_full_v2",
                        "--task", TASK, "--level", "0"],
                       cwd=MET, env=env, capture_output=True, text=True)
    txt = p.stdout + p.stderr
    open(f"{LOG}/{slug}__{gname}.log", "w").write(txt)
    official = {"n": len(GROUPS[gname]), "rc": p.returncode,
                "minutes": round((time.time() - t0) / 60, 1)}
    for k in PER_METRICS:
        h = re.findall(rf"(?:^|\s){k}:\s*([-\d.eE]+)", txt)
        if h:
            official[k] = float(h[-1])
    json.dump(official, open(f"{OUT}/{slug}__{gname}_official.json", "w"), indent=2)
    if p.returncode != 0:
        print(txt[-3000:])
    return official

q = queue.Queue()
for j in JOBS:
    if done(*j):
        print("already complete, skipping:", j)
    else:
        q.put(j)

def worker(gpu):
    while True:
        try:
            slug, gname = q.get_nowait()
        except queue.Empty:
            return
        print(f"[GPU{gpu}] START {slug} / {gname} (n={len(GROUPS[gname])})", flush=True)
        res = run(gpu, slug, gname)
        print(f"[GPU{gpu}] DONE  {slug} / {gname} -> {res}", flush=True)

threads = [threading.Thread(target=worker, args=(g,), daemon=True) for g in (0, 1)]
[t.start() for t in threads]
[t.join() for t in threads]
missing = [j for j in JOBS if not done(*j)]
assert not missing, f"jobs without complete per-image output: {missing} (see {LOG})"
print(f"✓ all metric jobs complete ({(time.time() - NB_START) / 60:.0f} min since notebook start)")


In [ ]:
# ===== 8. Validate the per-image values and build one per-case table =====
import json
import numpy as np
import pandas as pd

def load_job(slug, gname):
    frames = []
    for m in PER_METRICS:
        df = pd.read_csv(f"{OUT}/{slug}__{gname}_{m}.csv", dtype={"gen_path": str})
        parts = df["gen_path"].str.split("/")
        df["da_n"] = parts.str[-3]
        df["ins_id"] = parts.str[-2]
        df["case_id"] = parts.str[-1].str[:-4]
        assert not df.duplicated(["da_n", "ins_id", "case_id"]).any(), (slug, gname, m, "duplicate cases")
        frames.append(df.set_index(["da_n", "ins_id", "case_id"])[[m]])
    out = pd.concat(frames, axis=1)
    assert set(out.index) == set(GROUPS[gname]), (slug, gname, "case set differs from the group")
    assert out.notna().all().all(), (slug, gname, "missing values")
    return out

PER = {slug: load_job(slug, gname) for slug, gname in JOBS}
report = {"check3_internal_abs_diff": {}, "check4_published_table": {}, "rotate_identity_max_abs_diff": {}}

# Check 3 - the per-image values average exactly to the value printed by the official code in the same run.
for slug, gname in JOBS:
    off = json.load(open(f"{OUT}/{slug}__{gname}_official.json"))
    for m in PER_METRICS:
        dm = abs(float(PER[slug][m].mean()) - off[m])
        report["check3_internal_abs_diff"][f"{slug}/{gname}/{m}"] = dm
        assert dm < 1e-9, (slug, gname, m, float(PER[slug][m].mean()), off[m])
print("✓ check 3: per-image means reproduce the official printed values (|diff| < 1e-9)")

# Rotation images are byte-identical, so their per-image values must agree.
rot = GROUPS["rotate_1603"]
a = PER["FREEFINE_BASELINE"][PER["FREEFINE_BASELINE"].index.isin(rot)].sort_index()
b = PER["SGR_EPSREC"][PER["SGR_EPSREC"].index.isin(rot)].sort_index()
for m in PER_METRICS:
    report["rotate_identity_max_abs_diff"][m] = float((a[m] - b[m]).abs().max())
print("rotate identity, max |diff| per metric:", report["rotate_identity_max_abs_diff"])
assert report["rotate_identity_max_abs_diff"]["WRAP_E"] == 0.0
assert max(report["rotate_identity_max_abs_diff"].values()) < 1e-5

# Check 4 - group means reproduce the final metric table of the thesis (results/final_full5677_metrics.csv).
EXPECTED = {   # (SUBC, BGC, WRAP_E), stored rounded to 4 d.p.
    ("FREEFINE_BASELINE", "all_5677"): (0.9113, 0.9670, 0.0474),
    ("FREEFINE_BASELINE", "move_1439"): (0.9592, 0.9670, 0.0489),
    ("FREEFINE_BASELINE", "rotate_1603"): (0.9008, 0.9673, 0.0433),
    ("FREEFINE_BASELINE", "resize_2635"): (0.8916, 0.9669, 0.0490),
    ("FREEFINE_BASELINE", "resize_severe_759"): (0.7991, 0.9651, 0.0581),
    ("FREEFINE_BASELINE", "resize_nonsevere_1876"): (0.9290, 0.9676, 0.0453),
    ("SGR_EPSREC", "all_5677"): (0.9144, 0.9671, 0.0292),
    ("SGR_EPSREC", "move_1439"): (0.9665, 0.9672, 0.0121),
    ("SGR_EPSREC", "rotate_1603"): (0.9008, 0.9673, 0.0433),
    ("SGR_EPSREC", "resize_2635"): (0.8942, 0.9670, 0.0300),
    ("SGR_EPSREC", "resize_severe_759"): (0.7995, 0.9651, 0.0581),
    ("SGR_EPSREC", "resize_nonsevere_1876"): (0.9325, 0.9677, 0.0187),
    ("SGR_MIDHF_EPSREC", "resize_severe_759"): (0.7995, 0.9651, 0.0581),
}
fails = []
for (slug, g), exp in EXPECTED.items():
    if slug not in PER:
        continue
    sub = PER[slug][PER[slug].index.isin(GROUPS[g])]
    for m, e in zip(PER_METRICS, exp):
        got = round(float(sub[m].mean()), 4)
        report["check4_published_table"][f"{slug}/{g}/{m}"] = [got, e]
        if abs(got - e) > 1.1e-4:
            fails.append((slug, g, m, got, e))

# One per-case table: metadata + every per-image value.
base_cols = ["da_n", "ins_id", "case_id", "edit_type", "difficulty", "scale_exact", "affine_severe"]
wide = pd.DataFrame([{c: r[c] for c in base_cols} for r in meta]).set_index(["da_n", "ins_id", "case_id"])
for slug in PER:
    wide = wide.join(PER[slug].add_suffix(f"__{slug}"))
wide.reset_index().to_csv(f"{WORK}/percase_metrics_full5677.csv", index=False)
report["check4_failures"] = fails
json.dump(report, open(f"{WORK}/percase_validation_report.json", "w"), indent=2)
assert not fails, f"group means do not reproduce the final table: {fails}"
print("✓ check 4: group means reproduce the final metric table to 4 d.p.")
print("✓ saved percase_metrics_full5677.csv", wide.shape)


In [ ]:
# ===== 9. Paired statistics per group and metric =====
import numpy as np
import pandas as pd
from scipy.stats import wilcoxon

HIGHER_IS_BETTER = {"SUBC": True, "BGC": True, "WRAP_E": False}
TIE_TOL = 1e-6   # |delta| at or below this is a tie (identical images differ only by GPU round-off)
N_BOOT = 10000

def boot_ci(delta, n_boot=N_BOOT, seed=0, chunk=500):
    # percentile bootstrap of the mean paired difference, resampling cases
    rng = np.random.default_rng(seed)
    n = len(delta)
    means = np.empty(n_boot)
    for s in range(0, n_boot, chunk):
        k = min(chunk, n_boot - s)
        means[s:s + k] = delta[rng.integers(0, n, size=(k, n))].mean(axis=1)
    return np.percentile(means, [2.5, 97.5])

def paired(ref, new, metric):
    ref = np.asarray(ref, float)
    new = np.asarray(new, float)
    delta = new - ref
    delta = np.where(np.abs(delta) <= TIE_TOL, 0.0, delta)
    sign = 1.0 if HIGHER_IS_BETTER[metric] else -1.0
    better = int((sign * delta > 0).sum())
    worse = int((sign * delta < 0).sum())
    tie = len(delta) - better - worse
    if better + worse == 0:
        p, lo, hi = np.nan, 0.0, 0.0
    else:
        p = float(wilcoxon(delta, zero_method="wilcox", alternative="two-sided").pvalue)
        lo, hi = boot_ci(delta)
    sd = delta.std(ddof=1) if len(delta) > 1 else 0.0
    return {"n": len(delta), "mean_ref": ref.mean(), "mean_new": new.mean(), "delta": delta.mean(),
            "ci95_lo": lo, "ci95_hi": hi, "wilcoxon_p": p,
            "better": better, "tie": tie, "worse": worse,
            "frac_better": better / len(delta), "frac_worse": worse / len(delta),
            "d_z": (delta.mean() / sd) if sd > 0 else np.nan}

COMPARISONS = [("SGR_EPSREC", "FREEFINE_BASELINE", list(GROUPS))]
if INCLUDE_MIDHF_SEVERE:
    COMPARISONS += [("SGR_MIDHF_EPSREC", "FREEFINE_BASELINE", ["resize_severe_759"]),
                    ("SGR_MIDHF_EPSREC", "SGR_EPSREC", ["resize_severe_759"])]

rows = []
for new, ref, groups in COMPARISONS:
    for g in groups:
        sub = wide[wide.index.isin(GROUPS[g])]
        for m in PER_METRICS:
            r = paired(sub[f"{m}__{ref}"].values, sub[f"{m}__{new}"].values, m)
            rows.append({"comparison": f"{new} vs {ref}", "group": g, "metric": m, **r})
stats = pd.DataFrame(rows)
stats.to_csv(f"{WORK}/paired_stats.csv", index=False)

MAIN = ["all_5677", "move_1439", "rotate_1603", "resize_2635", "resize_severe_759", "resize_nonsevere_1876"]
show = stats[stats.group.isin(MAIN)].copy()
show["delta [95% CI]"] = show.apply(lambda r: f"{r.delta:+.4f} [{r.ci95_lo:+.4f}, {r.ci95_hi:+.4f}]", axis=1)
show["wilcoxon p"] = show.wilcoxon_p.map(lambda p: "-" if pd.isna(p) else f"{p:.1e}")
show["better/tie/worse"] = show.apply(lambda r: f"{r.better}/{r.tie}/{r.worse}", axis=1)
pd.set_option("display.width", 220)
print(show[["comparison", "group", "metric", "n", "delta [95% CI]", "wilcoxon p", "better/tie/worse"]]
      .to_string(index=False))
print("\n✓ saved paired_stats.csv", stats.shape)


In [ ]:
# ===== 10. Files to download =====
import os, time
for f in ["percase_metrics_full5677.csv", "paired_stats.csv", "percase_validation_report.json"]:
    p = f"{WORK}/{f}"
    print(f"{f:34s} {os.path.getsize(p) / 1e6:6.2f} MB")
print(f"total runtime {(time.time() - NB_START) / 60:.0f} min")


### After the run
Download the three files listed above (Output panel, or *Save Version* first so they persist)
and put them in `Diploma_Thesis_Writing/kaggle_runs/`.

Note for reading `paired_stats.csv`: on the move and non-severe-resize groups the warping error
of SGR-EPSREC is improved by construction (those branches paste the benchmark's coarse image,
which is also the warping-error reference). Its p-values there are real but are not evidence
of better editing.
